# VMChat FastAPI — Colab 临时测试部署

用途：临时测试 `vmchat-hermes-replacement-python`，把 FastAPI 暴露成一个临时公网 HTTPS 地址供本地 Vue 调用。

**注意：** Colab 运行时会断开，Cloudflare Quick Tunnel 地址也可能变化；这不是长期生产部署方案。


In [ ]:
# 1) 拉取最新 master 并安装依赖
!rm -rf /content/vmchat-hermes-replacement-python
!git clone -q https://github.com/liuruibing/vmchat-hermes-replacement-python.git /content/vmchat-hermes-replacement-python
%cd /content/vmchat-hermes-replacement-python
!python -m pip install -q uv
!uv sync --locked --no-dev --no-editable


In [ ]:
# 2) 安全输入模型配置（不会写进 GitHub，也不会回显 API Key）
import os, secrets, getpass

os.environ['LLM_PROVIDER'] = 'langchain'
os.environ['LLM_BASE_URL'] = input('LLM_BASE_URL: ').strip()
os.environ['LLM_API_KEY'] = getpass.getpass('LLM_API_KEY: ').strip()
model = input('LLM_MODEL [deepseek-v4-flash]: ').strip() or 'deepseek-v4-flash'
os.environ['LLM_MODEL'] = model
os.environ['LLM_OUTPUT_MODE'] = 'structured'
os.environ['CORS_ORIGINS'] = 'http://localhost:9528,http://127.0.0.1:9528'
os.environ['VMCHAT_SQL_KNOWLEDGE_PATH'] = 'delivery/02_vm_modules_sql_statements.md'
os.environ['MAX_SKILL_RESOURCE_READS'] = '24'
service_key = 'vmchat_' + secrets.token_urlsafe(24)
os.environ['SERVICE_API_KEY'] = service_key
print('配置完成。SERVICE_API_KEY 已生成；LLM_API_KEY 不会被打印。')


In [ ]:
# 3) 启动 FastAPI
import subprocess, time, requests, pathlib

for name in ('vmchat_uvicorn', 'vmchat_cloudflared'):
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()

log = open('/content/vmchat-uvicorn.log', 'w')
vmchat_uvicorn = subprocess.Popen(
    ['uv', 'run', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log, stderr=subprocess.STDOUT, env=os.environ.copy()
)

for _ in range(60):
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if r.status_code in (200, 503):
            print('FastAPI:', r.status_code, r.text)
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print(pathlib.Path('/content/vmchat-uvicorn.log').read_text()[-5000:])
    raise RuntimeError('FastAPI 启动失败')


In [ ]:
# 4) 安装 cloudflared 并创建临时 HTTPS Tunnel
import os, re, stat, time, subprocess, requests

cloudflared = '/content/cloudflared'
if not os.path.exists(cloudflared):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    os.chmod(cloudflared, os.stat(cloudflared).st_mode | stat.S_IEXEC)

tunnel_log_path = '/content/cloudflared.log'
tunnel_log = open(tunnel_log_path, 'w')
vmchat_cloudflared = subprocess.Popen(
    [cloudflared, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log, stderr=subprocess.STDOUT
)

public_url = None
for _ in range(60):
    time.sleep(1)
    tunnel_log.flush()
    text = open(tunnel_log_path, 'r', errors='ignore').read()
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    print(text[-5000:])
    raise RuntimeError('Cloudflare Tunnel 启动失败')

print('VMChat Base URL:', public_url)
print('Health URL:', public_url + '/health')
print('SERVICE_API_KEY:', service_key)
print('\n浏览器控制台执行：')
print(f"localStorage.setItem('fof-research-hermes-base-url', '{public_url}')")
print(f"localStorage.setItem('fof-research-hermes-api-key', '{service_key}')")
print("location.reload()")


In [ ]:
# 5) 可选：快速验证公网 health
import requests
resp = requests.get(public_url + '/health', timeout=15)
print(resp.status_code)
print(resp.text)


## 停止服务

Colab 断开后服务会自动消失。也可以手工运行下面的停止单元。


In [ ]:
for name in ('vmchat_cloudflared', 'vmchat_uvicorn'):
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()
print('已停止。')
